In [ ]:
# !wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py

In [1]:
import minsearch
import json

with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)


documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [2]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
) 

In [3]:
q = "the course has already started, can I still enroll?"

In [4]:
index.fit(documents)

In [25]:
boost = {'text': 3.0, 'question': 0.5}

results = index.search(
    query = q,
    filter_dict={'course': 'data-engineering-zoomcamp'},
    boost_dict = boost,
    num_results =1
)

In [7]:
results

'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'

In [6]:
from openai import OpenAI
import os
api_key = os.getenv("OPENROUTER_API_KEY")

In [8]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=api_key,
)

In [29]:
response = client.chat.completions.create(
  model="z-ai/glm-4.5-air:free", 
  messages=[
    {
      "role": "user",
      "content": prompt
    }
  ]
)

print(response.choices[0].message.content)

Yes, you can still enroll. 

Based solely on the CONTEXT, the answer to "The course has already started. Can I still join it?" is explicitly stated as "Yes, you can." The CONTEXT confirms enrollment is possible even after the course has started.


In [9]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [10]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT.
use only the facts from the CONTEXT when answering the QUESTION.
If the CONTEXT doesn't contain the answer, output NONE

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context = ""

    for doc in search_results:
        context = context + f"section: {doc['section']}\n question: {doc['question']}\n answer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()

    return prompt

In [11]:
def llm(prompt):
    response = client.chat.completions.create(
    model="z-ai/glm-4.5-air:free", 
    messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [12]:
query = 'how do i run  kafka'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)

    return answer

In [68]:
answer = rag('the course has already started, can I still enroll? Provide answer in refine version')
answer

'Based on the CONTEXT, here is the refined answer to whether you can still enroll after the course has started:\n\n**Yes, you can still enroll.**  \nYou will retain access to course materials and can participate in the course. However, you may miss the submission deadline for some earlier homework assignments. To earn a certificate, you must complete **2 out of 3 course projects** and **review 3 peers\' projects** by their respective deadlines. Latecomers joining by late November can still qualify for a certificate if they meet these requirements.  \n\n*(Supporting Context:  \n- "Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course."  \n- "Yes, even if you don’t register, you’re still eligible to submit the homeworks... deadlines for turning in the final projects."  \n- "if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.")*'

In [13]:
from elasticsearch import Elasticsearch

In [14]:
es_client = Elasticsearch('http://localhost:9200')

In [15]:
es_client.info()

ObjectApiResponse({'name': 'c8eab59ff5e0', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'iXvPZJurQF6EgEQDUEteyQ', 'version': {'number': '8.4.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '42f05b9372a9a4a470db3b52817899b99a76ee73', 'build_date': '2022-10-04T07:17:24.662462378Z', 'build_snapshot': False, 'lucene_version': '9.3.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [16]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"

es_client.indices.create(index=index_name, body=index_settings)

BadRequestError: BadRequestError(400, 'resource_already_exists_exception', 'index [course-questions/3rOqTQ8kTyWOn-cE9X5umQ] already exists')

In [17]:
from tqdm.auto import tqdm

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
for dov in tqdm(documents):
    es_client.index(index=index_name, document=doc)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 948/948 [00:03<00:00, 239.75it/s]


In [19]:
query = "I just discovered the course. can I still join it?"

In [20]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)
    
    result_docs = []
    
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [45]:
result = elastic_search('I just discovered the course. can I still join it?')
result

[]

In [26]:
def rag(query):
    search_results = elastic_search("I just discovered the course. can I still join it?")
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [27]:
rag(query)

"Based on the provided CONTEXT, the answer is **NONE**.\n\nThe CONTEXT does not contain any information about the course's enrollment policies, deadlines, or whether late registration is permitted. Therefore, it is not possible to determine if you can still join the course based solely on the given information."